## Pulling EPMT DB

This notebook is trying to pull data from the epmt database and put them into csv files. We're chunking them because I seem to have trouble holding data in memory over 40k rows. I was able to save over 100k rows this way, so I'll stick to this for now.

In [ ]:
# Environment work around to get matplotlib/seaborn
import sys
sys.path.append('/home/Janice.Kim/work/epmt/misc/jk_py37')

In [ ]:
import epmt_query as eq
import orm.sqlalchemy.models as models
from pprint import pprint
from datetime import datetime
from pathlib import Path
import csv
import pandas as pd

In [ ]:
# Query how many rows exist
eq.get_jobs(fmt='orm').count()

In [ ]:
# Get the number of jobs from the past 21 days
eq.get_jobs(fmt='orm', after=-21).count()

In [ ]:
one=eq.get_jobs(fmt='dict', after=-21, limit=1)[0]
pprint(one['write_bytes'])

## Let's collect all of these rows

In [ ]:
data = eq.get_jobs(fmt='orm', after=-21)

## Actually, Let's Query the DB in Chunks
Instead of getting 50k rows all in one go, let's query the db in chunks.... Let's query the db in 5k chunks and write those chunks to file. Let's use the jobid to help us order the results and make sure we don't miss or duplicate anything...

In [ ]:
#tags = ['jobid', 'env_dict', 'read_bytes', 'write_bytes', 'time_waiting','cpu_time', 'duration', 'start', 'end', 'minflt', 'majflt', 'tags', 'annotations', 'all_proc_tags', 'exitcode']
tags = ['jobid', 'env_dict', 'annotations', 'exitcode', 'cpu_time', 'duration', 'start', 'end', 'tags']

In [ ]:
data.count()

In [ ]:
type(data)

In [ ]:
chunk_size = 5000

total_rows = data.count()
print(f"Processing {total_rows} rows.")

row_count = 0
file_count = 0
csv_file = None
writer = None

date_str = datetime.today().strftime("%Y%m%d_%H%M%S")

work_dir = Path(f"/home/Janice.Kim/work/epmt/data_{total_rows}_{date_str}")
work_dir.mkdir(parents=True, exist_ok=True)

for row in data.yield_per(1000):
    #print(row_count)
    #print(type(row))

    if row_count == 0:
        if csv_file:
            csv_file.close()

        filepath = work_dir / f"data_{total_rows}_{file_count}_{date_str}.csv"
        print(f"Writing to {filepath}")
        
        csv_file = open(filepath, "w", encoding="utf-8") 

        writer = csv.DictWriter(csv_file, fieldnames=tags)
        writer.writeheader()
        
    # Convert ORM obj to dict
    row_dict = {col: getattr(row, col) for col in tags}
    #print(row_dict)
    writer.writerow(row_dict)

    row_count += 1
    if row_count >= chunk_size:
        row_count = 0
        file_count += 1
        if (file_count % 10 == 0):
            input(f"Reached end of file {file_count}...")


Now that we have it in a file, can we clean it up and make it useful? - TODO in another notebook